In [10]:
# -*- coding: utf-8 -*-
"""
Edge AI Challenge 2026 - Training and Export Script
Huấn luyện mô hình phân loại biển báo giao thông với QAT,
xuất file .h5, .tflite int8, và tạo file submission.csv
"""

import os
# BẮT BUỘC: Đặt biến môi trường trước khi import tensorflow
os.environ['TF_USE_LEGACY_KERAS'] = '1'

import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping, ModelCheckpoint
import tensorflow_model_optimization as tfmot

# ------------------------- CẤU HÌNH -------------------------
DATASET_PATH = "C:/DUT/Ki_2/Edge_AI/dataset"
TRAIN_DIR = os.path.join(DATASET_PATH, "train")
TEST_DIR = os.path.join(DATASET_PATH, "test")
IMG_SIZE = (32, 32)          # 32x32 pixels
BATCH_SIZE = 64
NUM_CLASSES = 10             # Giả sử 10 lớp (0-9)
EPOCHS_FLOAT = 30            # Số epoch huấn luyện float32
EPOCHS_QAT = 10              # Số epoch fine-tune với QAT
MODEL_H5_PATH = "best_model.h5"
MODEL_TFLITE_PATH = "model_int8.tflite"
SUBMISSION_CSV = "submission.csv"

# ------------------------- 1. TẢI DỮ LIỆU -------------------------
print("=== 1. Loading data ===")

# Tạo generator với augmentation cho train
train_datagen = ImageDataGenerator(
    rescale=1./255,                     # Chuẩn hóa về [0,1] cho float32
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    shear_range=0.1,
    zoom_range=0.1,
    brightness_range=[0.8, 1.2],
    horizontal_flip=False,              # Tùy chỉnh nếu biển đối xứng
    fill_mode='nearest',
    validation_split=0.2                # Giữ lại 20% làm validation
)

# Generator cho validation (không augmentation)
val_datagen = ImageDataGenerator(rescale=1./255, validation_split=0.2)

# Load ảnh từ thư mục train (các thư mục con là tên class)
train_generator = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training',
    shuffle=True
)

validation_generator = val_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation',
    shuffle=False
)

# Lấy mapping class -> index (0..9)
class_indices = train_generator.class_indices
print("Class indices:", class_indices)

# ------------------------- 2. XÂY DỰNG MÔ HÌNH -------------------------
print("\n=== 2. Building model ===")

def create_model(input_shape=(32,32,3), num_classes=10):
    inputs = layers.Input(shape=input_shape)
    
    # Conv2D đầu tiên (stride 2 để giảm kích thước)
    x = layers.Conv2D(16, (3,3), strides=2, padding='same', use_bias=False)(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU(6.0)(x)
    
    # Depthwise separable convolutions
    x = layers.SeparableConv2D(32, (3,3), padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU(6.0)(x)
    
    x = layers.SeparableConv2D(64, (3,3), strides=2, padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU(6.0)(x)
    
    x = layers.SeparableConv2D(128, (3,3), padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU(6.0)(x)
    
    x = layers.SeparableConv2D(256, (3,3), strides=2, padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU(6.0)(x)
    
    # Global average pooling thay vì Flatten + Dense dày
    x = layers.GlobalAveragePooling2D()(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)
    
    model = models.Model(inputs, outputs)
    return model

# Tạo mô hình
model = create_model(input_shape=(32,32,3), num_classes=NUM_CLASSES)
print(f"Total parameters: {model.count_params():,}")

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# ------------------------- 3. HUẤN LUYỆN FLOAT32 -------------------------
print("\n=== 3. Training float32 model ===")

callbacks_float = [
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, verbose=1),
    EarlyStopping(monitor='val_loss', patience=8, restore_best_weights=True),
    ModelCheckpoint('best_float.h5', save_best_only=True, monitor='val_accuracy')
]

history_float = model.fit(
    train_generator,
    steps_per_epoch=train_generator.samples // BATCH_SIZE,
    epochs=EPOCHS_FLOAT,
    validation_data=validation_generator,
    validation_steps=validation_generator.samples // BATCH_SIZE,
    callbacks=callbacks_float
)

# Load best weights (nếu đã được lưu)
if os.path.exists('best_float.h5'):
    model.load_weights('best_float.h5')
    print("Loaded best float32 weights.")

# ------------------------- 4. QUANTIZATION AWARE TRAINING (QAT) -------------------------
print("\n=== 4. Quantization Aware Training ===")

# Áp dụng QAT lên mô hình đã huấn luyện
qat_model = tfmot.quantization.keras.quantize_model(model)

# Compile lại với learning rate nhỏ hơn để fine-tune
qat_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-4),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Fine-tune
callbacks_qat = [
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, verbose=1),
    EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True),
    ModelCheckpoint('best_qat.h5', save_best_only=True, monitor='val_accuracy')
]

history_qat = qat_model.fit(
    train_generator,
    steps_per_epoch=train_generator.samples // BATCH_SIZE,
    epochs=EPOCHS_QAT,
    validation_data=validation_generator,
    validation_steps=validation_generator.samples // BATCH_SIZE,
    callbacks=callbacks_qat
)

# Lưu mô hình QAT dạng .h5 (yêu cầu của BTC)
qat_model.save(MODEL_H5_PATH)
print(f"Saved QAT model to {MODEL_H5_PATH}")

# ------------------------- 5. CHUYỂN ĐỔI SANG TFLITE INT8 -------------------------
print("\n=== 5. Converting to TFLite int8 ===")

# Hàm tạo representative dataset (dùng 100 ảnh từ tập train để xác định khoảng giá trị)
def representative_dataset():
    # Lấy một số ảnh từ validation_generator (float32 [0,1]) và chuyển về uint8 [0,255]
    # Để đơn giản, lấy 10 batch đầu
    for i, (images, _) in enumerate(validation_generator):
        images_uint8 = (images * 255).astype(np.uint8)
        for img in images_uint8:
            yield [np.expand_dims(img, axis=0)]
        if i >= 10:  # Lấy 10 batch * batch_size = 640 ảnh, đủ
            break

# Chuyển đổi
converter = tf.lite.TFLiteConverter.from_keras_model(qat_model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_dataset
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.uint8
converter.inference_output_type = tf.uint8

tflite_quant_model = converter.convert()

# Lưu file
with open(MODEL_TFLITE_PATH, 'wb') as f:
    f.write(tflite_quant_model)
print(f"Saved TFLite int8 model to {MODEL_TFLITE_PATH}")

# ------------------------- 6. DỰ ĐOÁN TRÊN TẬP TEST -------------------------
print("\n=== 6. Predicting on test set ===")

# Tải mô hình TFLite
interpreter = tf.lite.Interpreter(model_path=MODEL_TFLITE_PATH)
interpreter.allocate_tensors()
input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

# Lấy danh sách file ảnh test
test_files = sorted([f for f in os.listdir(TEST_DIR) if f.lower().endswith('.png')])
# Hàm dự đoán
def predict_tflite(image_path):
    img = keras.preprocessing.image.load_img(image_path, target_size=IMG_SIZE)
    img_array = keras.preprocessing.image.img_to_array(img)  # uint8
    # Mô hình yêu cầu đầu vào uint8, không cần chuẩn hóa
    img_input = np.expand_dims(img_array, axis=0).astype(np.uint8)
    interpreter.set_tensor(input_details[0]['index'], img_input)
    interpreter.invoke()
    output = interpreter.get_tensor(output_details[0]['index'])
    return np.argmax(output[0])

# Dự đoán và ghi file CSV
submission_data = []
for file_name in test_files:
    img_id = os.path.splitext(file_name)[0]  # lấy phần tên không có .png
    img_path = os.path.join(TEST_DIR, file_name)
    pred_label = predict_tflite(img_path)
    submission_data.append([img_id, pred_label])

# Ghi CSV
import csv
with open(SUBMISSION_CSV, 'w', newline='') as csvfile:
    writer = csv.writer(csvfile)
    writer.writerow(['Id', 'Label'])
    writer.writerows(submission_data)

print(f"Submission file saved to {SUBMISSION_CSV}")

print("\n=== All tasks completed successfully! ===")

ModuleNotFoundError: No module named 'tf_keras'